# Road Damage Detection - YOLO26n Fine-Tuning (RDD)Trains `yolo26n` on the Road Damage Detection dataset, then exports weights for CPUinference on a local machine.**Before you run anything:** `Runtime -> Change runtime type -> T4 GPU`. Without this youare training on a CPU in the cloud, which is slower than your own laptop.**Expected result:** mAP50 around **0.55-0.65**. That is a normal, respectable score onRDD - the dataset has small, ambiguous, heavily-occluded targets and inconsistent labelsbetween countries. Treat anything near 0.9 as a sign you are validating on your trainingdata, not as success.**Runtime:** roughly 2-4 hours for 60 epochs on a T4. Start it and go build the app.**Damage classes**| ID | Code | Name ||:--|:--|:--|| 0 | D00 | Longitudinal Crack || 1 | D10 | Transverse Crack || 2 | D20 | Alligator Crack || 3 | D40 | Pothole |

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.")

## 2. Install Ultralytics

In [ ]:
%pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

## 3. Mount Google DriveColab free tier disconnects without warning. Writing run output to Drive means a droppedsession costs you the time since the last checkpoint, not the whole run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/road-damage-training'
RUN_NAME = 'yolo26n_rdd'
Path(PROJECT_DIR).mkdir(parents=True, exist_ok=True)
print("Run output ->", f"{PROJECT_DIR}/{RUN_NAME}")

## 4. Get the datasetRDD2022 ships as PASCAL VOC XML, which YOLO cannot read. Converting it yourself is ahalf-day of fiddling, so the primary path here pulls a pre-converted YOLO-format copyfrom Roboflow Universe.**Setup (2 minutes):**1. Make a free account at [roboflow.com](https://roboflow.com)2. Search Universe for a road damage / RDD dataset with the 4 classes above3. `Download this Dataset` -> format **YOLOv8** -> `show download code`4. Paste your API key and the workspace/project/version values belowThe YOLOv8 export format is what Ultralytics expects - it works unchanged for YOLO26.

In [ ]:
from roboflow import Roboflow

# ---- fill these in from the Roboflow download snippet ----
ROBOFLOW_API_KEY = ""      # e.g. "abc123..."
WORKSPACE        = ""      # e.g. "my-workspace"
PROJECT          = ""      # e.g. "road-damage-detection"
VERSION          = 1
# ----------------------------------------------------------

if not ROBOFLOW_API_KEY:
    raise SystemExit("Fill in ROBOFLOW_API_KEY, WORKSPACE, PROJECT, VERSION first.")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = (
    rf.workspace(WORKSPACE)
      .project(PROJECT)
      .version(VERSION)
      .download("yolov8", location="/content/rdd")
)

DATA_YAML = f"{dataset.location}/data.yaml"
print("data.yaml ->", DATA_YAML)

### Fallback: convert raw RDD2022 yourselfOnly run this if you could not find a usable pre-converted dataset. It downloads the rawRDD2022 archive and converts PASCAL VOC XML to YOLO txt labels.Keep the dataset in Colab. Do not sync it to Drive and do not download it locally - it isroughly 10GB and local disk is the tightest constraint on the target machine.

In [ ]:
# FALLBACK PATH - skip if the Roboflow cell above worked.
import os, shutil, random
import xml.etree.ElementTree as ET
from pathlib import Path

RUN_FALLBACK = False   # flip to True to use this path

CLASSES = ["D00", "D10", "D20", "D40"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}


# Convert one VOC XML annotation to YOLO format. Returns number of boxes written.
def voc_to_yolo(xml_path: Path, out_path: Path) -> int:
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    w, h = int(size.find("width").text), int(size.find("height").text)
    if w == 0 or h == 0:
        return 0

    lines = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        if name not in CLASS_TO_ID:
            continue                      # RDD has extra classes we ignore
        bb = obj.find("bndbox")
        x1, y1 = float(bb.find("xmin").text), float(bb.find("ymin").text)
        x2, y2 = float(bb.find("xmax").text), float(bb.find("ymax").text)
        x1, x2 = max(0, min(x1, x2)), min(w, max(x1, x2))
        y1, y2 = max(0, min(y1, y2)), min(h, max(y1, y2))
        if x2 <= x1 or y2 <= y1:
            continue
        lines.append(
            f"{CLASS_TO_ID[name]} {((x1+x2)/2)/w:.6f} {((y1+y2)/2)/h:.6f} "
            f"{(x2-x1)/w:.6f} {(y2-y1)/h:.6f}"
        )

    out_path.write_text("\n".join(lines))
    return len(lines)


if RUN_FALLBACK:
    !mkdir -p /content/rdd_raw && cd /content/rdd_raw && \
        wget -q --show-progress https://mycityreport.s3-ap-northeast-1.amazonaws.com/02_RoadDamageDataset/mycity/RDD2022/RDD2022_Japan.zip && \
        unzip -q RDD2022_Japan.zip

    src_img = Path("/content/rdd_raw/Japan/train/images")
    src_ann = Path("/content/rdd_raw/Japan/train/annotations/xmls")
    root = Path("/content/rdd")

    pairs = [(p, src_ann / f"{p.stem}.xml") for p in sorted(src_img.glob("*.jpg"))]
    pairs = [(i, a) for i, a in pairs if a.exists()]
    random.seed(0)
    random.shuffle(pairs)
    split = int(len(pairs) * 0.8)

    for name, subset in (("train", pairs[:split]), ("valid", pairs[split:])):
        (root / name / "images").mkdir(parents=True, exist_ok=True)
        (root / name / "labels").mkdir(parents=True, exist_ok=True)
        for img, ann in subset:
            shutil.copy(img, root / name / "images" / img.name)
            voc_to_yolo(ann, root / name / "labels" / f"{img.stem}.txt")
        print(f"{name}: {len(subset)} images")

    (root / "data.yaml").write_text(
        f"path: {root}\ntrain: train/images\nval: valid/images\n"
        f"nc: {len(CLASSES)}\nnames: {CLASSES}\n"
    )
    DATA_YAML = str(root / "data.yaml")
    print("data.yaml ->", DATA_YAML)

## 5. Sanity-check the dataset before burning GPU hours

In [ ]:
import yaml
from pathlib import Path
from collections import Counter

cfg = yaml.safe_load(open(DATA_YAML))
print(yaml.dump(cfg, sort_keys=False))

base = Path(cfg.get("path", Path(DATA_YAML).parent))
for split in ("train", "val"):
    rel = cfg.get(split)
    if not rel:
        continue
    img_dir = (base / rel).resolve()
    lbl_dir = Path(str(img_dir).replace("images", "labels"))
    imgs = list(img_dir.glob("*.*"))
    lbls = list(lbl_dir.glob("*.txt"))
    counts = Counter()
    for l in lbls:
        for line in l.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    print(f"{split:5s}  {len(imgs):6d} images  {len(lbls):6d} labels")
    for cid in sorted(counts):
        print(f"         class {cid} ({cfg['names'][cid]:22s}) {counts[cid]:6d} boxes")

assert cfg["nc"] == 4, f"Expected 4 classes, got {cfg['nc']} - check the dataset version"

# ---------------------------------------------------------------------------
# Class ORDER must match src/utils/constants.py, which maps id -> damage type
# by position. A dataset that lists the same four classes in a different order
# trains perfectly and scores well, then mislabels every detection in the app.
# Nothing raises. This is the check that catches it.
# ---------------------------------------------------------------------------
EXPECTED = ["D00", "D10", "D20", "D40"]
ALIASES = {
    "d00": "D00", "longitudinal": "D00", "longitudinal crack": "D00",
    "d10": "D10", "transverse": "D10", "transverse crack": "D10",
    "d20": "D20", "alligator": "D20", "alligator crack": "D20",
                  "aligator crack": "D20",
    "d40": "D40", "pothole": "D40", "potholes": "D40",
}

actual = []
for raw in cfg["names"]:
    key = str(raw).strip().lower().replace("_", " ").replace("-", " ")
    actual.append(ALIASES.get(key, f"UNKNOWN({raw})"))

print("\nClass order in this dataset:")
for i, (raw, code) in enumerate(zip(cfg["names"], actual)):
    flag = "  <-- MISMATCH" if code != EXPECTED[i] else ""
    print(f"  {i}: {raw:28s} -> {code}{flag}")

if actual == EXPECTED:
    print("\nOK - order matches src/utils/constants.py. Safe to train.")
else:
    print(f"\n*** ORDER MISMATCH ***")
    print(f"  dataset:  {actual}")
    print(f"  expected: {EXPECTED}")
    print("\nFix ONE of these before training:")
    print("  a) pick a dataset ordered D00, D10, D20, D40, or")
    print("  b) reorder DAMAGE_CLASSES in src/utils/constants.py to match the")
    print("     dataset above, keeping class_id 0,1,2,3 sequential.")
    print("\nTraining now would produce a model that mislabels every detection.")

## 6. Train`patience=15` stops early once validation mAP plateaus, so a run that has learnedeverything it will learn does not keep burning your Colab quota.`save_period=5` writes a checkpoint to Drive every 5 epochs for disconnect recovery.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data=DATA_YAML,
    epochs=60,
    imgsz=640,
    batch=16,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    patience=15,        # early stop when val mAP plateaus
    save_period=5,      # checkpoint to Drive for disconnect recovery
    workers=2,          # Colab throttles above this
    optimizer="auto",
    seed=0,
    plots=True,
)

### If the session droppedRe-run cells 1-5 to restore the environment, then run this instead of the cell above.Ultralytics picks up from the last checkpoint on Drive.

In [ ]:
# RECOVERY ONLY - run instead of the training cell above after a disconnect.
from ultralytics import YOLO

RESUME = False   # flip to True after a dropped session

if RESUME:
    model = YOLO(f"{PROJECT_DIR}/{RUN_NAME}/weights/last.pt")
    results = model.train(resume=True)

## 7. Evaluate

In [ ]:
from ultralytics import YOLO

BEST = f"{PROJECT_DIR}/{RUN_NAME}/weights/best.pt"
model = YOLO(BEST)

metrics = model.val(data=DATA_YAML, imgsz=640, split="val")

print(f"\nmAP50    : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"precision: {metrics.box.mp:.4f}")
print(f"recall   : {metrics.box.mr:.4f}")

print("\nPer class:")
for i, name in enumerate(model.names.values()):
    print(f"  {name:24s} mAP50={metrics.box.ap50[i]:.4f}")

print("\nCopy these numbers into the README results table - including the weak ones.")

In [ ]:
# Training curves and confusion matrix - screenshot these for the README.
from IPython.display import Image, display
import os

for f in ("results.png", "confusion_matrix_normalized.png", "val_batch0_pred.jpg"):
    p = f"{PROJECT_DIR}/{RUN_NAME}/{f}"
    if os.path.exists(p):
        print(f"\n=== {f} ===")
        display(Image(filename=p, width=900))

## 8. Export for CPU inferenceONNX runs substantially faster than PyTorch on the target CPU (Ryzen 5 5625U, no NVIDIAGPU). `simplify=True` folds constants and cleans up the graph.`opset=12` is chosen for broad `onnxruntime` compatibility - raise it only if you hit anunsupported-operator error locally.

In [ ]:
from ultralytics import YOLO

model = YOLO(BEST)
onnx_path = model.export(format="onnx", imgsz=640, simplify=True, opset=12, dynamic=False)
print("ONNX ->", onnx_path)

## 9. Download the weightsTake **only** these two files - a few MB total. The dataset stays in Colab; local disk onthe target machine has under 15GB free.Place both in `D:\place proj\road-damage-detector\models\`.

In [ ]:
from google.colab import files
import shutil, os

weights_dir = f"{PROJECT_DIR}/{RUN_NAME}/weights"
shutil.copy(f"{weights_dir}/best.pt", "/content/road_damage_best.pt")

onnx_src = f"{weights_dir}/best.onnx"
if os.path.exists(onnx_src):
    shutil.copy(onnx_src, "/content/road_damage_best.onnx")

for f in ("/content/road_damage_best.pt", "/content/road_damage_best.onnx"):
    if os.path.exists(f):
        print(f"{f}  ({os.path.getsize(f)/1e6:.1f} MB)")
        files.download(f)

---## DoneBoth files also live in Drive at `road-damage-training/yolo26n_rdd/weights/` if thebrowser download fails.**Next:** drop them into `models/`, then run `python scripts/benchmark.py` to get thePyTorch-vs-ONNX timing table for the README.